# 02 Statistics figures

Extended Data Fig. 2 panels and the five-method consensus, read from and written to `StatResults`.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "scripts")))
from paths import RESULTS

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Paths
# -----------------------------
BASE = os.path.join(RESULTS, "StatResults")
OUT  = os.path.join(BASE, "SlidesFigures_More")
os.makedirs(OUT, exist_ok=True)

# -----------------------------
# Small helpers
# -----------------------------
def _load_csv(path, **kwargs):
    try:
        return pd.read_csv(path, **kwargs)
    except Exception as e:
        print(f"[skip] {os.path.basename(path)} not loaded: {e}")
        return None

def savefig(fname):
    plt.tight_layout()
    path = os.path.join(OUT, fname)
    plt.savefig(path, dpi=220, bbox_inches="tight")
    plt.close()
    print(f"[ok] {fname}")

def pretty_targets(cols):
    mapping = {
        "Y0":"qc", "Y1":"kc",
        "Y2":"q(kc/6)", "Y3":"q(kc/3)", "Y4":"q(kc/2)",
        "Y5":"q(2kc/3)", "Y6":"q(5kc/6)",
        "Y7":"flow_pca1","Y8":"flow_pca2","Y9":"flow_pca3"
    }
    return [mapping.get(c, c) for c in cols]

def ensure_numeric(df):
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# Professional label casing for human-readable labels (keeps VIP/VIF all-caps)
def _titlecase_label(s):
    if not isinstance(s, str):
        return s
    # replace underscores with spaces for nicer tick labels
    s = s.replace("_", " ")
    words = s.split()
    out = []
    for w in words:
        if w.upper() in {"VIP", "VIF"}:
            out.append(w.upper())
        else:
            out.append(w[:1].upper() + w[1:] if w else w)
    return " ".join(out)

# -----------------------------
# 1) PLS2 plots
# -----------------------------
# Find any pls2_* files; assume a single tag (e.g., "all_targets")
pls_r2_files  = glob.glob(os.path.join(BASE, "pls2_*_r2.csv"))
pls_vip_files = glob.glob(os.path.join(BASE, "pls2_*_vip.csv"))
pls_coef_files= glob.glob(os.path.join(BASE, "pls2_*_coefficients.csv"))

if pls_r2_files:
    r2_df = _load_csv(pls_r2_files[0])
    if r2_df is not None and {"response","r2_in_sample"}.issubset(r2_df.columns):
        # Order by Y0..Y9 if present; else keep file order
        order = sorted(r2_df["response"], key=lambda x: (x[0]!="Y", int(x[1:]) if x[1:].isdigit() else 999))
        r2_df = r2_df.set_index("response").loc[order].reset_index()
        xs = np.arange(len(r2_df))
        plt.figure(figsize=(max(7, 0.65*len(xs)), 4.5))
        plt.bar(xs, r2_df["r2_in_sample"].values)
        plt.xticks(xs, pretty_targets(r2_df["response"].tolist()), rotation=30, ha="right")
        plt.ylabel("R² (In-Sample)")
        plt.title("PLS2 — In-Sample R² by Response")
        for xi, v in zip(xs, r2_df["r2_in_sample"].values):
            if np.isfinite(v):
                plt.text(xi, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
        savefig("20_pls2_r2_bars.png")
else:
    print("[note] No PLS2 R² file found; skipping PLS2 R² plot.")

if pls_vip_files:
    vip_df = _load_csv(pls_vip_files[0], index_col=0)
    if vip_df is not None and vip_df.shape[0] > 0:
        vip = vip_df.iloc[:,0].astype(float) if vip_df.shape[1]==1 else vip_df.iloc[:,0].astype(float)
        vip = vip.sort_values(ascending=False)
        top_n = min(25, vip.shape[0])
        plt.figure(figsize=(max(7, 0.5*top_n), 4.5))
        plt.bar(np.arange(top_n), vip.iloc[:top_n].values)
        plt.xticks(np.arange(top_n), vip.index[:top_n], rotation=60, ha="right")
        plt.ylabel("VIP")
        plt.title(f"PLS2 — VIP (Top {top_n})")
        savefig("21_pls2_vip_topN.png")
else:
    print("[note] No PLS2 VIP file found; skipping VIP plot.")

if pls_coef_files:
    coef_df = _load_csv(pls_coef_files[0], index_col=0)
    if coef_df is not None and coef_df.shape[0] > 0:
        # Use L2 across targets for ranking features
        coefs = ensure_numeric(coef_df.copy())
        l2 = np.sqrt((coefs**2).sum(axis=1)).sort_values(ascending=False)
        top_n = min(25, l2.shape[0])
        sub = coefs.loc[l2.index[:top_n]]
        plt.figure(figsize=(max(7, 0.55*top_n), max(4.5, 0.4*sub.shape[1])))
        im = plt.imshow(sub.values, aspect="auto", interpolation="nearest")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        plt.yticks(np.arange(sub.shape[0]), sub.index)
        plt.xticks(np.arange(sub.shape[1]), pretty_targets(sub.columns.tolist()), rotation=30, ha="right")
        plt.title(f"PLS2 — Coefficients Heatmap (Top {top_n} by L2)")
        savefig("22_pls2_coefficients_heatmap_topN.png")
else:
    print("[note] No PLS2 coefficient file found; skipping coefficient heatmap.")

# -----------------------------
# 2) Elastic Net plots
# -----------------------------
enet_std_files = glob.glob(os.path.join(BASE, "elasticnet_*_coefficients_standardizedX.csv"))
enet_orig_files= glob.glob(os.path.join(BASE, "elasticnet_*_coefficients_original_units.csv"))
enet_hyp_files = glob.glob(os.path.join(BASE, "elasticnet_*_intercepts_and_hyperparams.csv"))

if enet_std_files:
    en_std = _load_csv(enet_std_files[0], index_col=0)
    if en_std is not None and en_std.shape[0] > 0:
        en_std = ensure_numeric(en_std)
        # Rank features by L2 across targets (standardized => comparable)
        l2 = np.sqrt((en_std**2).sum(axis=1)).sort_values(ascending=False)
        top_n = min(25, l2.shape[0])
        sub = en_std.loc[l2.index[:top_n]]
        plt.figure(figsize=(max(7, 0.55*top_n), max(4.5, 0.4*sub.shape[1])))
        im = plt.imshow(sub.values, aspect="auto", interpolation="nearest")
        plt.colorbar(im, fraction=0.046, pad=0.04)
        plt.yticks(np.arange(sub.shape[0]), sub.index)
        plt.xticks(np.arange(sub.shape[1]), pretty_targets(sub.columns.tolist()), rotation=30, ha="right")
        plt.title(f"Elastic Net — Standardized-X Coefficients Heatmap (Top {top_n} by L2)")
        savefig("23_elasticnet_coefficients_standardized_heatmap_topN.png")

        # Per-target bars for key targets if present
        def bar_top_abs_for_target(col_name, nice_name):
            if col_name not in en_std.columns: 
                return False
            s = en_std[col_name].abs().sort_values(ascending=False)
            k = min(20, s.shape[0])
            plt.figure(figsize=(max(7, 0.45*k), 4.5))
            plt.bar(np.arange(k), s.iloc[:k].values)
            plt.xticks(np.arange(k), s.index[:k], rotation=60, ha="right")
            plt.ylabel("|Coef| (Standardized X)")
            plt.title(f"Elastic Net — Top |Coef| for {nice_name}")
            savefig(f"24_elasticnet_top20_abscoef_{nice_name}.png")
            return True

        any_target_plot = False
        # Try both raw Y names and pretty aliases
        target_map = {
            "Y0":"qc", "Y1":"kc", "Y7":"flow_pca1"
        }
        for raw, nice in target_map.items():
            if raw in en_std.columns:
                any_target_plot |= bar_top_abs_for_target(raw, nice)
        # If aliases exist instead of Y*, try them directly too
        for alias in ["qc","kc","flow_pca1"]:
            if alias in en_std.columns:
                any_target_plot |= bar_top_abs_for_target(alias, alias)
        if not any_target_plot:
            print("[note] Elastic Net: target columns not found for per-target bar plots; skipped.")

else:
    print("[note] No Elastic Net standardized coefficient file found; skipping EN heatmap/bars.")

if enet_hyp_files:
    hyp = _load_csv(enet_hyp_files[0])
    if hyp is not None and {"response","alpha","l1_ratio"}.issubset(hyp.columns):
        # Alpha distribution
        plt.figure(figsize=(6.5,4.0))
        plt.hist(pd.to_numeric(hyp["alpha"], errors="coerce").dropna().values, bins=15)
        plt.xlabel("Chosen Alpha")
        plt.ylabel("Count")
        plt.title("Elastic Net — Distribution of Chosen Alpha")
        savefig("25_elasticnet_alpha_distribution.png")

        # l1_ratio distribution
        plt.figure(figsize=(6.5,4.0))
        plt.hist(pd.to_numeric(hyp["l1_ratio"], errors="coerce").dropna().values, bins=10)
        plt.xlabel("Chosen L1 Ratio")
        plt.ylabel("Count")
        plt.title("Elastic Net — Distribution of Chosen L1 Ratio")
        savefig("26_elasticnet_l1ratio_distribution.png")
else:
    print("[note] No Elastic Net hyperparameter file found; skipping alpha/l1_ratio plots.")

# -----------------------------
# 3) Correlation + FDR plots
# -----------------------------
corr_files = sorted(glob.glob(os.path.join(BASE, "corr_predictors_vs_*.csv")))
if corr_files:
    sig_counts = []
    for fp in corr_files:
        df = _load_csv(fp)
        if df is None or df.empty:
            continue
        # Expect columns: predictor, rho, p_value, n, is_significant_bh_0.05, q_bh
        target = os.path.splitext(os.path.basename(fp))[0].replace("corr_predictors_vs_","")
        if "rho" not in df.columns: 
            continue
        df["abs_rho"] = df["rho"].abs()
        if "q_bh" in df.columns:
            df["neglog10_q"] = -np.log10(np.where(df["q_bh"]<=0, np.nan, df["q_bh"]))
        else:
            # derive approx q via rank if missing
            pvals = pd.to_numeric(df.get("p_value", pd.Series(np.nan, index=df.index)), errors="coerce")
            n = pvals.shape[0]
            order = np.argsort(pvals.values)
            ranks = np.empty_like(order); ranks[order] = np.arange(1, n+1)
            q = pvals.values * n / ranks
            # monotone
            q_sorted = q[order]
            for i in range(n-2, -1, -1):
                q_sorted[i] = min(q_sorted[i], q_sorted[i+1])
            q_adj = np.empty_like(q); q_adj[order] = q_sorted
            df["q_bh"] = q_adj
            df["neglog10_q"] = -np.log10(np.where(df["q_bh"]<=0, np.nan, df["q_bh"]))

        # volcano-style: |rho| vs -log10(q_bh)
        plt.figure(figsize=(6.6, 5))
        sig_mask = df.get("is_significant_bh_0.05", (df["q_bh"] < 0.05))
        x = df["abs_rho"].values
        y = df["neglog10_q"].values
        plt.scatter(x[~sig_mask], y[~sig_mask], s=14, alpha=0.6, label="NS")
        plt.scatter(x[sig_mask],  y[sig_mask],  s=16, alpha=0.9, label="BH q<0.05")
        plt.xlabel("|Spearman Rho|")
        plt.ylabel("-log10(qBH)")
        plt.title(f"Correlations vs {target}\n(|Rho| vs -log10 q)")
        plt.legend()
        fname = f"30_corr_volcano_{target}.png"
        savefig(fname)

        # histogram of |rho|
        plt.figure(figsize=(6.4,4.2))
        plt.hist(df["abs_rho"].dropna().values, bins=25)
        plt.xlabel("|Rho|")
        plt.ylabel("Count")
        plt.title(f"|Rho| Distribution — {target}")
        savefig(f"31_corr_absrho_hist_{target}.png")

        # tally significant
        sig_counts.append((target, int(sig_mask.sum()), int(sig_mask.shape[0])))

    if sig_counts:
        sig_df = pd.DataFrame(sig_counts, columns=["target","n_significant","n_total"]).sort_values("target")
        sig_df["frac"] = sig_df["n_significant"] / sig_df["n_total"].replace(0, np.nan)
        xs = np.arange(sig_df.shape[0])
        plt.figure(figsize=(max(7, 0.6*len(xs)), 4.5))
        plt.bar(xs, sig_df["n_significant"].values)
        plt.xticks(xs, sig_df["target"].tolist(), rotation=30, ha="right")
        plt.ylabel("# Features with q<0.05")
        plt.title("Count of Significant Correlations per Target (BH q<0.05)")
        for xi, v in zip(xs, sig_df["n_significant"].values):
            plt.text(xi, v, str(v), ha="center", va="bottom", fontsize=8)
        savefig("32_corr_significant_counts_per_target.png")
else:
    print("[note] No correlation tables (corr_predictors_vs_*.csv); skipping correlation plots.")

# -----------------------------
# 4) Diagnostics plots
# -----------------------------
diag_dir = os.path.join(BASE, "diagnostics")
drop_sum_fp = os.path.join(diag_dir, "feature_drop_summary.csv")
vif_fp      = os.path.join(diag_dir, "final_vif.csv")
kept_fp     = os.path.join(diag_dir, "predictors_kept.csv")
removed_fp  = os.path.join(diag_dir, "predictors_removed_total.csv")

drop_df = _load_csv(drop_sum_fp)
if drop_df is not None and {"step","n_after","dropped"}.issubset(drop_df.columns):
    steps = drop_df["step"].tolist()
    # Make x-axis step labels Title Case (VIP/VIF preserved)
    steps_pretty = [_titlecase_label(s) for s in steps]
    remaining = pd.to_numeric(drop_df["n_after"], errors="coerce").values
    plt.figure(figsize=(max(7, 0.65*len(steps)), 4.4))
    plt.bar(np.arange(len(steps)), remaining)
    plt.xticks(np.arange(len(steps)), steps_pretty, rotation=0)
    plt.ylabel("Remaining Features")
    plt.title("Feature Selection")
    for i, v in enumerate(remaining):
        if np.isfinite(v):
            plt.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=8)
    savefig("40_feature_drop_counts_by_step.png")

else:
    print("[note] Diagnostics feature_drop_summary.csv not found; skipping drop-count plot.")

vif_df = _load_csv(vif_fp, header=None)
if vif_df is not None and not vif_df.empty:
    # Expect [feature, VIF] or a Series; handle both
    if vif_df.shape[1] == 2:
        vif_vals = pd.to_numeric(vif_df.iloc[:,1], errors="coerce").dropna().values
    else:
        vif_vals = pd.to_numeric(vif_df.iloc[:,0], errors="coerce").dropna().values
    plt.figure(figsize=(6.5,4.2))
    plt.hist(vif_vals, bins=25)
    plt.xlabel("VIF")
    plt.ylabel("Count")
    plt.title("Final VIF Distribution (Kept Predictors)")
    savefig("41_final_vif_histogram.png")
    if vif_vals.size:
        pct_le4 = 100.0 * (vif_vals <= 4.0).mean()
        with open(os.path.join(OUT, "41_final_vif_summary.txt"), "w", encoding="utf-8") as fh:
            fh.write(f"Count: {vif_vals.size}\n")
            fh.write(f"Median VIF: {np.nanmedian(vif_vals):.2f}\n")
            fh.write(f"IQR VIF: {np.nanpercentile(vif_vals, 75) - np.nanpercentile(vif_vals, 25):.2f}\n")
            fh.write(f"% with VIF <= 4: {pct_le4:.1f}%\n")
else:
    print("[note] Diagnostics final_vif.csv not found; skipping VIF plots.")

kept_df    = _load_csv(kept_fp, header=None, names=["feature"])
removed_df = _load_csv(removed_fp, header=None, names=["feature"])
if kept_df is not None and removed_df is not None:
    counts = [kept_df.shape[0], removed_df.shape[0]]
    labels = ["Kept", "Removed"]
    plt.figure(figsize=(6.0,4.2))
    plt.bar(np.arange(2), counts)
    plt.xticks(np.arange(2), labels)
    plt.ylabel("# Features")
    plt.title("Kept vs Removed Predictors")
    for i, v in enumerate(counts):
        plt.text(i, v, str(int(v)), ha="center", va="bottom", fontsize=10)
    savefig("42_kept_vs_removed_features.png")

# -----------------------------
# 5) Random-Forest permutation importance (per-target CSVs)
# -----------------------------
rf_perm_files = sorted(glob.glob(os.path.join(BASE, "rf_permutation_importance_*.csv")))
if rf_perm_files:
    for fp in rf_perm_files:
        tgt = os.path.splitext(os.path.basename(fp))[0].replace("rf_permutation_importance_","")
        df = _load_csv(fp)
        if df is None or {"predictor","perm_importance_mean"}.issubset(df.columns) is False:
            continue
        df = df.sort_values("perm_importance_mean", ascending=False)
        k = min(15, df.shape[0])
        plt.figure(figsize=(max(7, 0.5*k), 4.5))
        plt.bar(np.arange(k), df["perm_importance_mean"].iloc[:k].values)
        plt.xticks(np.arange(k), df["predictor"].iloc[:k].tolist(), rotation=60, ha="right")
        plt.ylabel("Permutation Importance (Mean)")
        plt.title(f"Random Forest Permutation Importance — {tgt} (Top {k})")
        savefig(f"50_rf_permutation_importance_top_{tgt}.png")
else:
    print("[note] No rf_permutation_importance_*.csv files found; skipping RF permutation plots.")

# -----------------------------
# 6) Flow PCA (heatmap variant, complements earlier line plots)
# -----------------------------
pca_load_fp = os.path.join(BASE, "flow_pca_loadings.csv")
pca_scores_fp = os.path.join(BASE, "flow_pca_scores.csv")
pca_expl_fp = os.path.join(BASE, "flow_pca_explained_variance.csv")

load_df = _load_csv(pca_load_fp)
if load_df is not None and not load_df.empty:
    df = load_df.copy()
    # Expect columns to be flow names; rows are components (index "component" or 0..n-1)
    # Reindex columns in the intended order if present:
    col_order = ["q_kc/6","q_kc/3","q_kc/2","q_2kc/3","q_5kc/6","q_kc"]
    cols = [c for c in col_order if c in df.columns] + [c for c in df.columns if c not in col_order]
    df = df[cols]
    plt.figure(figsize=(7.5, 4.5))
    im = plt.imshow(df.values, aspect="auto", interpolation="nearest")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.yticks(np.arange(df.shape[0]), [f"PC{i+1}" for i in range(df.shape[0])])
    plt.xticks(np.arange(df.shape[1]), df.columns, rotation=30, ha="right")
    plt.title("Flow PCA — Loadings Heatmap")
    savefig("60_flow_pca_loadings_heatmap.png")
else:
    print("[note] flow_pca_loadings.csv not found; skipping loadings heatmap.")

scores_df = _load_csv(pca_scores_fp, index_col=0)
if scores_df is not None and set(["flow_pca1","flow_pca2"]).issubset(scores_df.columns):
    plt.figure(figsize=(6.0,5.2))
    plt.scatter(scores_df["flow_pca1"].values, scores_df["flow_pca2"].values, s=14, alpha=0.8)
    plt.xlabel("PC1 Score")
    plt.ylabel("PC2 Score")
    plt.title("Flow PCA — PC1 vs PC2 Scores")
    savefig("61_flow_pca_scores_scatter_pc1_pc2.png")

expl_df = _load_csv(pca_expl_fp)
if expl_df is not None and {"component","explained_variance_ratio"}.issubset(expl_df.columns):
    xs = np.arange(expl_df.shape[0]) + 1
    plt.figure(figsize=(6.0,4.2))
    plt.plot(xs, expl_df["explained_variance_ratio"].values, marker="o")
    plt.xticks(xs, [f"PC{i}" for i in xs])
    plt.ylabel("Explained Variance Ratio")
    plt.title("Flow PCA — Scree (From CSV)")
    savefig("62_flow_pca_scree_from_csv.png")

print("Done. Extra figures saved to:", OUT)

In [ ]:
import os, re, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------- CONFIG --------------------
OUT_DIR   = os.path.join(RESULTS, "StatResults")
DIAG_DIR  = os.path.join(OUT_DIR, "diagnostics")
os.makedirs(DIAG_DIR, exist_ok=True)

# Thresholds / controls
VIP_THRESHOLD        = 1.0
EN_ZERO_TOL          = 1e-8
SPEARMAN_ALPHA_QBH   = 0.05
SPEARMAN_ABS_RHO_MIN = 0.30
RF_Z_CUTOFF          = 0.0   # 0.0 -> mean > 0; 1.0 -> mean > 1*std; 2.0 -> mean > 2*std
MIN_COUNT_TO_PLOT    = 2     # show only features that >= this many methods agree on
TOP_N_TO_PLOT        = 25    # limit bars for readability (set None for all)

# Targets (7 “original outputs”). File-safe names must match your saved files.
TARGET_MAP = {
    "qc":        {"corr": "qc",       "lasso": "qc",       "rf": "qc",       "en_col": "Y0"},
    "kc":        {"corr": "kc",       "lasso": "kc",       "rf": "kc",       "en_col": "Y1"},
    "q_kc_6":    {"corr": "q_kc_6",   "lasso": "q_kc_6",   "rf": "q_kc_6",   "en_col": "Y2"},
    "q_kc_3":    {"corr": "q_kc_3",   "lasso": "q_kc_3",   "rf": "q_kc_3",   "en_col": "Y3"},
    "q_kc_2":    {"corr": "q_kc_2",   "lasso": "q_kc_2",   "rf": "q_kc_2",   "en_col": "Y4"},
    "q_2kc_3":   {"corr": "q_2kc_3",  "lasso": "q_2kc_3",  "rf": "q_2kc_3",  "en_col": "Y5"},
    "q_5kc_6":   {"corr": "q_5kc_6",  "lasso": "q_5kc_6",  "rf": "q_5kc_6",  "en_col": "Y6"},
}

# File patterns
PATHS = {
    "vip": os.path.join(OUT_DIR, "pls2_all_targets_vip.csv"),
    "enet": os.path.join(OUT_DIR, "elasticnet_all_targets_coefficients_original_units.csv"),
    "corr_glob": os.path.join(OUT_DIR, "corr_predictors_vs_*.csv"),
    "rf_glob": os.path.join(OUT_DIR, "rf_permutation_importance_*.csv"),
    "lasso_glob": os.path.join(OUT_DIR, "lasso_ols_*.txt"),
}

# -------------------- LOADERS --------------------
def load_vip(vip_path):
    vip_ok = set()
    if os.path.exists(vip_path):
        df = pd.read_csv(vip_path)
        # Support both “feature,VIP” and index-as-feature shapes
        if "VIP" in df.columns and "Unnamed: 0" in df.columns:
            df = df.rename(columns={"Unnamed: 0": "feature"})
        if "VIP" in df.columns and "feature" in df.columns:
            vip_ok = set(df.loc[df["VIP"] >= VIP_THRESHOLD, "feature"].astype(str))
        else:
            # Fallback: treat first column as feature, second as VIP
            fcol = df.columns[0]
            vcol = df.columns[1]
            vip_ok = set(df.loc[df[vcol] >= VIP_THRESHOLD, fcol].astype(str))
    return vip_ok

def load_elasticnet(enet_path):
    """Returns: dict target_en_col -> set(features with |coef|>tol) and the full DF."""
    out = {}
    df = None
    if os.path.exists(enet_path):
        df = pd.read_csv(enet_path)
        # Expect first col is feature name or index; ensure we have a feature column:
        if df.columns[0] != "feature":
            df = df.rename(columns={df.columns[0]: "feature"})
        df["feature"] = df["feature"].astype(str)
        for _, meta in TARGET_MAP.items():
            col = meta["en_col"]
            if col in df.columns:
                mask = df[col].abs() > EN_ZERO_TOL
                out[col] = set(df.loc[mask, "feature"].astype(str))
    return out, df

def load_corrs(corr_glob):
    """Return per-target sets based on (q_bh <= alpha) & (|rho| >= min)."""
    per_target = {k: set() for k in TARGET_MAP.keys()}
    for fp in glob.glob(corr_glob):
        name = os.path.splitext(os.path.basename(fp))[0]  # corr_predictors_vs_<tgt>
        tgt_fs = name.replace("corr_predictors_vs_", "")
        try:
            df = pd.read_csv(fp)
        except Exception:
            continue
        # normalize predictor column name
        if "predictor" not in df.columns:
            # best guess: first col
            df = df.rename(columns={df.columns[0]: "predictor"})
        # thresholds
        if "q_bh" in df.columns and "rho" in df.columns:
            mask = (df["q_bh"] <= SPEARMAN_ALPHA_QBH) & (df["rho"].abs() >= SPEARMAN_ABS_RHO_MIN)
            sig = set(df.loc[mask, "predictor"].astype(str))
        else:
            sig = set()
        # map back to our canonical target key if present
        for key, meta in TARGET_MAP.items():
            if tgt_fs == meta["corr"]:
                per_target[key] |= sig
                break
    return per_target

def load_rf(rf_glob):
    """Return per-target sets using mean > RF_Z_CUTOFF*std (std may be 0)."""
    per_target = {k: set() for k in TARGET_MAP.keys()}
    for fp in glob.glob(rf_glob):
        name = os.path.splitext(os.path.basename(fp))[0]  # rf_permutation_importance_<tgt>
        tgt_fs = name.replace("rf_permutation_importance_", "")
        try:
            df = pd.read_csv(fp)
        except Exception:
            continue
        # normalize columns
        if "predictor" not in df.columns:
            df = df.rename(columns={df.columns[0]: "predictor"})
        mcol = "perm_importance_mean"
        scol = "perm_importance_std"
        if mcol in df.columns:
            if scol in df.columns:
                thr_mask = df[mcol] > RF_Z_CUTOFF * df[scol]
            else:
                thr_mask = df[mcol] > 0
            sig = set(df.loc[thr_mask, "predictor"].astype(str))
        else:
            sig = set()
        for key, meta in TARGET_MAP.items():
            if tgt_fs == meta["rf"]:
                per_target[key] |= sig
                break
    return per_target

def parse_lasso_selected(text_path):
    """Parse 'Selected features:' list from lasso_ols_<tgt>.txt → set(features)."""
    sel = []
    try:
        with open(text_path, "r", encoding="utf-8") as f:
            txt = f.read()
    except Exception:
        return set()
    # grab lines after 'Selected features:' until a blank or next section
    m = re.search(r"Selected features:\s*(.*?)\n\s*\n", txt, flags=re.DOTALL)
    if not m:
        # fallback: read bullet lines anywhere
        m = re.search(r"Selected features:\s*(.*)", txt, flags=re.DOTALL)
    if m:
        block = m.group(1)
        for line in block.splitlines():
            line = line.strip()
            if line.startswith("-"):
                feat = line[1:].strip()
                if feat:
                    sel.append(feat)
    return set(sel)

def load_lasso(lasso_glob):
    """Return per-target sets from the 'Selected features' sections."""
    per_target = {k: set() for k in TARGET_MAP.keys()}
    for fp in glob.glob(lasso_glob):
        name = os.path.splitext(os.path.basename(fp))[0]  # lasso_ols_<tgt>
        tgt_fs = name.replace("lasso_ols_", "")
        sel = parse_lasso_selected(fp)
        for key, meta in TARGET_MAP.items():
            if tgt_fs == meta["lasso"]:
                per_target[key] |= sel
                break
    return per_target

# -------------------- AGGREGATION --------------------
# Load all method signals
vip_ok = load_vip(PATHS["vip"])
enet_per_col, enet_df = load_elasticnet(PATHS["enet"])
corr_sig = load_corrs(PATHS["corr_glob"])
rf_sig   = load_rf(PATHS["rf_glob"])
la_sig   = load_lasso(PATHS["lasso_glob"])

# ElasticNet per-target: map EN column name -> our target key
enet_per_target = {k: enet_per_col.get(meta["en_col"], set()) for k, meta in TARGET_MAP.items()}

# Overall agreement (by method, ANY target)
overall_flags = {}
all_features = set()
methods = ["VIP", "ElasticNet", "Spearman", "RF", "LASSO"]

# Build overall method-wise sets
overall_sets = {
    "VIP": vip_ok,
    "ElasticNet": set().union(*enet_per_target.values()) if len(enet_per_target) else set(),
    "Spearman": set().union(*corr_sig.values()) if len(corr_sig) else set(),
    "RF": set().union(*rf_sig.values()) if len(rf_sig) else set(),
    "LASSO": set().union(*la_sig.values()) if len(la_sig) else set(),
}
for s in overall_sets.values():
    all_features |= s

# Make overall table
overall_tbl = pd.DataFrame({"feature": sorted(all_features)})
for m in methods:
    overall_tbl[m] = overall_tbl["feature"].isin(overall_sets[m]).astype(int)
overall_tbl["count_methods"] = overall_tbl[methods].sum(axis=1)
overall_tbl = overall_tbl.sort_values(["count_methods","feature"], ascending=[False,True])

# Save overall CSV
overall_csv = os.path.join(DIAG_DIR, "consensus_overall.csv")
overall_tbl.to_csv(overall_csv, index=False)

# Plot overall
to_plot = overall_tbl[overall_tbl["count_methods"] >= MIN_COUNT_TO_PLOT]
if TOP_N_TO_PLOT is not None:
    to_plot = to_plot.head(TOP_N_TO_PLOT)

plt.figure(figsize=(10, max(4, 0.35*len(to_plot))))
plt.barh(to_plot["feature"], to_plot["count_methods"])
plt.xlabel("Number of methods agreeing (VIP, EN, Spearman, RF, LASSO)")
plt.title("Consensus (overall): important features agreed by multiple methods")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(DIAG_DIR, "consensus_overall.png"), dpi=200)
plt.close()

# -------------------- PER-OUTPUT FIGURES --------------------
def build_per_output_table(tgt_key):
    """For a single target, build [feature, LASSO, RF, Spearman, ElasticNet, count_methods]."""
    rows = []
    # gather sets
    en = enet_per_target.get(tgt_key, set())
    sp = corr_sig.get(tgt_key, set())
    rf = rf_sig.get(tgt_key, set())
    la = la_sig.get(tgt_key, set())
    # union of all features flagged by any method for this output
    feats = set().union(en, sp, rf, la)
    if not feats:
        return pd.DataFrame(columns=["feature","LASSO","RF","Spearman","ElasticNet","count_methods"])
    df = pd.DataFrame({"feature": sorted(feats)})
    df["LASSO"] = df["feature"].isin(la).astype(int)
    df["RF"] = df["feature"].isin(rf).astype(int)
    df["Spearman"] = df["feature"].isin(sp).astype(int)
    df["ElasticNet"] = df["feature"].isin(en).astype(int)
    df["count_methods"] = df[["LASSO","RF","Spearman","ElasticNet"]].sum(axis=1)
    return df.sort_values(["count_methods","feature"], ascending=[False,True])

# Iterate targets and emit plot + CSV
for tgt_key in TARGET_MAP.keys():
    pretty_name = tgt_key.replace("_", " ").replace("kc", "k_c")  # optional prettifier
    tbl = build_per_output_table(tgt_key)
    csv_path = os.path.join(DIAG_DIR, f"consensus_{tgt_key}.csv")
    tbl.to_csv(csv_path, index=False)

    sub = tbl[tbl["count_methods"] >= MIN_COUNT_TO_PLOT]
    if TOP_N_TO_PLOT is not None:
        sub = sub.head(TOP_N_TO_PLOT)

    plt.figure(figsize=(10, max(4, 0.35*len(sub))))
    plt.barh(sub["feature"], sub["count_methods"])
    plt.xlabel("Number of methods agreeing (LASSO, RF, Spearman, ElasticNet)")
    plt.title(f"Consensus for {tgt_key}: important features agreed by multiple methods")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(DIAG_DIR, f"consensus_{tgt_key}.png"), dpi=200)
    plt.close()

print("Done. Wrote:")
print(f"- Overall: {overall_csv} and consensus_overall.png")
print(f"- Per-output CSV/PNG for: {', '.join(TARGET_MAP.keys())}")